# 00 - Data Checks and Preliminary Spatial Characterisation

This notebook documents the initial data loading, coordinate system verification, and preliminary spatial characterisation.

In [3]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

## 1. Load dataset and inspect CRS

In [4]:
features = gpd.read_file("/Users/amber/Documents/M Thesis/Data/QGIS/isaura_features.gpkg")

print(f"CRS: {features.crs}")
print(f"Total features: {len(features)}")
print(f"\nFeature class counts:")
print(features['type'].value_counts())

CRS: EPSG:32644
Total features: 7243

Feature class counts:
type
hydrology    3911
mound        2671
void          469
temple        192
Name: count, dtype: int64


## 2. Load parcel boundary and verify area

The parcel boundary is extracted from the Archaeoscape `parcels.gpkg` file by filtering on the `index` column for the value `08_isaura`. The area computed directly by GeoPandas from the resulting polygon serves as the primary verification of the coordinate unit.

In [6]:
parcels = gpd.read_file("/Users/amber/Documents/M Thesis/Data/parcels.gpkg")
parcel  = parcels[parcels['index'] == '08_isaura'].copy().reset_index(drop=True)

print(f"CRS: {parcel.crs}")
print(f"Parcel area (GeoPandas): {parcel.geometry.area.values[0] / 1e6:.4f} km²")
print(f"Known parcel extent:     182.6 km²")

CRS: EPSG:32644
Parcel area (GeoPandas): 182.5972 km²
Known parcel extent:     182.6 km²


## 3. Confirm coordinate unit

The coordinate unit is confirmed by comparing the parcel polygon area in native units against the known parcel extent. No conversion factor is applied to the area calculation - the match between the two values directly establishes the unit.

In [8]:
parcel_area_native = parcel.geometry.area.values[0]  # in native units²
known_area_m2      = 182.6e6                         # 182.6 km² in m²

native_unit_m = np.sqrt(known_area_m2 / parcel_area_native)

print(f"Parcel area (native units²): {parcel_area_native:,.2f}")
print(f"Known area (m²):             {known_area_m2:,.0f}")
print(f"\n1 native unit = {native_unit_m:.6f} metres")

Parcel area (native units²): 182,597,227.50
Known area (m²):             182,600,000

1 native unit = 1.000008 metres


## 4. Filter mound features and extract centroids

Only mound polygons are used in the clustering analysis. Centroids are extracted from each polygon geometry, reducing each feature to a single representative point.

In [10]:
mounds = features[features['type'] == 'mound'].copy()
print(f"Mound features: {len(mounds)}")

mounds = mounds.copy()
mounds['cx'] = mounds.geometry.centroid.x
mounds['cy'] = mounds.geometry.centroid.y

coords = mounds[['cx', 'cy']].values
print(f"Centroid array shape: {coords.shape}")
print(f"\nX range: {coords[:, 0].min():.1f} – {coords[:, 0].max():.1f} m")
print(f"Y range: {coords[:, 1].min():.1f} – {coords[:, 1].max():.1f} m")

Mound features: 2671
Centroid array shape: (2671, 2)

X range: 364.3 – 21384.9 m
Y range: -4.7 – 15655.6 m


## 5. Preliminary spatial characterisation - nearest-neighbour distances

Nearest-neighbour distances (NND) are computed using a BallTree with Euclidean metric. The 1st, 5th, and 10th nearest-neighbour distances characterise local grouping structure at progressively coarser scales. The mean 1st-NND (120 m) defines the lower bound for the DBSCAN epsilon range and the finest KDE bandwidth tested.

In [11]:
tree = BallTree(coords, metric='euclidean')

# Query 1st, 5th, and 10th nearest neighbours (k+1 to exclude self)
distances, _ = tree.query(coords, k=11)

nnd_1  = distances[:, 1]   # 1st nearest neighbour
nnd_5  = distances[:, 5]   # 5th nearest neighbour
nnd_10 = distances[:, 10]  # 10th nearest neighbour

print("1st nearest-neighbour distance (m):")
print(f"  Mean:   {nnd_1.mean():.1f}")
print(f"  Median: {np.median(nnd_1):.1f}")
print(f"  Min:    {nnd_1.min():.1f}")
print(f"  Max:    {nnd_1.max():.1f}")
print(f"  Std:    {nnd_1.std():.1f}")

print("\n5th nearest-neighbour distance (m):")
print(f"  Mean:   {nnd_5.mean():.1f}")
print(f"  Median: {np.median(nnd_5):.1f}")

print("\n10th nearest-neighbour distance (m):")
print(f"  Mean:   {nnd_10.mean():.1f}")
print(f"  Median: {np.median(nnd_10):.1f}")

1st nearest-neighbour distance (m):
  Mean:   120.1
  Median: 103.3
  Min:    13.5
  Max:    1541.3
  Std:    83.5

5th nearest-neighbour distance (m):
  Mean:   271.7
  Median: 242.5

10th nearest-neighbour distance (m):
  Mean:   395.4
  Median: 361.7
